### Module 5: Embeddings and Vector Stores

**What You'll Learn**
- How to convert text chunks into numerical embeddings using OpenAI.
- How to store embeddings in a vector database (Chroma).
- How to perform similarity search to retrieve relevant chunks.

#### Key Concepts
- **Embedding**: a list of numbers representing the meaning of text.
- **Vector Store**: a database that stores embeddings and supports fast similarity search.

**Diagram**
Chunks → Embedding Model → Vectors → Store in Chroma
Query → Embedding Model → Query Vector → Similarity Search → Top-k Chunks



In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

Step 2: Raw Text → Document → Chunks

In [2]:
# Define raw sample text
raw_text = '''
RAG, or Retrieval-Augmented Generation, is a technique that combines information retrieval with language generation. 
It retrieves relevant documents from a knowledge base and uses them as context for an LLM to generate grounded answers. 
This helps reduce hallucinations and keeps the model's knowledge up-to-date.

Vector stores are specialised databases that store embeddings and enable fast similarity search. 
They allow us to find the most relevant chunks for a given query by comparing vector distances. 
Common vector stores include Chroma, FAISS, and Pinecone.

LangChain provides tools for building RAG pipelines easily. 
It integrates document loaders, text splitters, embedding models, and vector stores into one workflow. 
This simplifies the process of going from raw files to a fully functional retrieval system.
'''


In [3]:
# Convert raw text into a LangChain Document object
doc = Document(
    page_content=raw_text,
    metadata={'source': 'sample.txt'}
)

In [4]:
# Create a text splitter with chunk size and overlap
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=['\n\n', '\n', '.', ' ', '']
)

# Split the document into chunks
chunks = splitter.split_documents([doc])

print(f'Created {len(chunks)} chunks from raw texts.')

for i, chunk in enumerate(chunks, start=1):
    print(f'\nChunk {i}: {chunk.page_content}')

Created 6 chunks from raw texts.

Chunk 1: RAG, or Retrieval-Augmented Generation, is a technique that combines information retrieval with language generation.

Chunk 2: It retrieves relevant documents from a knowledge base and uses them as context for an LLM to generate grounded answers. 
This helps reduce hallucinations and keeps the model's knowledge up-to-date.

Chunk 3: Vector stores are specialised databases that store embeddings and enable fast similarity search. 
They allow us to find the most relevant chunks for a given query by comparing vector distances.

Chunk 4: Common vector stores include Chroma, FAISS, and Pinecone.

Chunk 5: LangChain provides tools for building RAG pipelines easily. 
It integrates document loaders, text splitters, embedding models, and vector stores into one workflow.

Chunk 6: This simplifies the process of going from raw files to a fully functional retrieval system.


Step 3: Embed Chunks and Store in Chroma

In [5]:
# Create the embedding model
embeddings = OpenAIEmbeddings()

# Create a Chroma vector store from the chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=None  # In-memory only (lost when process exits)
)

print('Chunks embedded and stored in Chroma Successfully.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chunks embedded and stored in Chroma Successfully.


Generate the embedding vector for this text

In [6]:
# Get the first chunk from the list
sample_chunk = chunks[0]
print('Chosen text:')
print(sample_chunk.page_content)
print('---')

# Generate the embedding vector
vector = embeddings.embed_query(sample_chunk.page_content)

print(f'Embedding vector length: {len(vector)} numbers')

# Print the first 10 numbers of the vector
print('\nFirst 10 numbers:')
print(vector[:10])

Chosen text:
RAG, or Retrieval-Augmented Generation, is a technique that combines information retrieval with language generation.
---
Embedding vector length: 1536 numbers

First 10 numbers:
[-0.045473698526620865, -0.0017859070794656873, 0.011257417500019073, -0.008906681090593338, 0.0051912106573581696, 0.01066973339766264, -0.0211305133998394, -0.012785396538674831, -0.04374982416629791, -0.0460222028195858]


Step 4: Perform a Similarity Search

In [7]:
# Define a user query
query = 'How do vector stores help in RAG?'

# Search the vector store for the 2 most similar chunks
results = vectorstore.similarity_search(query, k=2)

print(f'Top {len(results)} relevant chunks for query: "{query}"\n')

for i, doc in enumerate(results, start=1):
    print(f'Chunk {i}: {doc.page_content}')
    print(f'Metadata: {doc.metadata}')
    print('-' * 60)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Top 2 relevant chunks for query: "How do vector stores help in RAG?"

Chunk 1: Vector stores are specialised databases that store embeddings and enable fast similarity search. 
They allow us to find the most relevant chunks for a given query by comparing vector distances.
Metadata: {'source': 'sample.txt'}
------------------------------------------------------------
Chunk 2: RAG, or Retrieval-Augmented Generation, is a technique that combines information retrieval with language generation.
Metadata: {'source': 'sample.txt'}
------------------------------------------------------------


#### Step 1: Load Chunks from JSON

In [8]:
import json
from pathlib import Path
from langchain_core.documents import Document

# Define the path to the saved chunks
chunks_file = Path('../04_data_ingestion_document_processing/data/chunks/all_chunks.json')

# Load the JSON data
with open(chunks_file, 'r', encoding='utf-8') as f:
    chunk_data = json.load(f)
    
print(f'Loaded {len(chunk_data)} chunks from JSON')

chunk_data


Loaded 312 chunks from JSON


[{'content': 'Agriculture in Nigeria: Crops, Livestock, and Disease Management\n\n\n\nAgriculture in Nigeria\nLast updated: August 2026\n\n\nIntroduction',
  'metadata': {'source': '04_data_ingestion_document_processing\\data\\agriculture.html',
   'title': 'Agriculture in Nigeria: Crops, Livestock, and Disease Management',
   'file_name': 'agriculture.html',
   'doc_type': 'html',
   'language': 'English',
   'chunk_id': 'agriculture.html_001'}},
 {'content': "Introduction\n\n            Agriculture is one of the most important sectors of the Nigerian economy. It contributes significantly to the country's GDP and employs more than a third of the labour force. In rural areas, agriculture is the primary source of income and food security.",
  'metadata': {'source': '04_data_ingestion_document_processing\\data\\agriculture.html',
   'title': 'Agriculture in Nigeria: Crops, Livestock, and Disease Management',
   'file_name': 'agriculture.html',
   'doc_type': 'html',
   'language': 'Engli

Step 2: Convert to Document Objects

In [9]:
real_chunks = []

for item in chunk_data:
    doc = Document(
        page_content=item['content'],
        metadata=item['metadata']
    )
    
    real_chunks.append(doc)

print(f'Created {len(real_chunks)} Document objects.')
print(real_chunks)

Created 312 Document objects.
[Document(metadata={'source': '04_data_ingestion_document_processing\\data\\agriculture.html', 'title': 'Agriculture in Nigeria: Crops, Livestock, and Disease Management', 'file_name': 'agriculture.html', 'doc_type': 'html', 'language': 'English', 'chunk_id': 'agriculture.html_001'}, page_content='Agriculture in Nigeria: Crops, Livestock, and Disease Management\n\n\n\nAgriculture in Nigeria\nLast updated: August 2026\n\n\nIntroduction'), Document(metadata={'source': '04_data_ingestion_document_processing\\data\\agriculture.html', 'title': 'Agriculture in Nigeria: Crops, Livestock, and Disease Management', 'file_name': 'agriculture.html', 'doc_type': 'html', 'language': 'English', 'chunk_id': 'agriculture.html_002'}, page_content="Introduction\n\n            Agriculture is one of the most important sectors of the Nigerian economy. It contributes significantly to the country's GDP and employs more than a third of the labour force. In rural areas, agriculture

Step 3: Embed and Store Real Chunks in Chroma

In [10]:
# Create embedding
embeddings = OpenAIEmbeddings()

# Create a Chroma vector store
vectorstore = Chroma.from_documents(
    documents=real_chunks,
    embedding=embeddings,
    persist_directory=None
)

print('Real chunks embedded and stored in Chroma.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Real chunks embedded and stored in Chroma.


Step 4: Visualise Vectors of First 3 Real Chunks


In [11]:
embeddings = OpenAIEmbeddings()

# Select the first 3 real chunks
#sample_chunk = real_chunks[:3]

# Get embedding vectors from first 3 real chunks
sample_vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in real_chunks[:3]]
)

sample_vectors

[[-0.007216715719550848,
  -0.02362428605556488,
  -0.010589745827019215,
  -0.011132307350635529,
  -0.007942309603095055,
  0.004566011019051075,
  -0.023166703060269356,
  0.012086691334843636,
  -0.04664717614650726,
  -0.021963918581604958,
  -0.015204783529043198,
  0.004118234384804964,
  -0.0025444806087762117,
  0.0124789047986269,
  0.01813330687582493,
  0.012733843177556992,
  0.03074948489665985,
  -0.03273669630289078,
  0.003451472846791148,
  -0.0182640440762043,
  -0.029991207644343376,
  0.02750719152390957,
  0.005383120849728584,
  0.003363224910572171,
  0.02240842580795288,
  0.0053210207261145115,
  0.00587338674813509,
  -0.03080178052186966,
  0.02014666423201561,
  -0.004807875491678715,
  -0.013380994088947773,
  -0.010713947005569935,
  0.009347738698124886,
  -0.007863867096602917,
  -0.01728351227939129,
  -0.029128339141607285,
  0.010092942975461483,
  -0.006942166946828365,
  0.03250136971473694,
  0.010145238600671291,
  0.03506382554769516,
  0.010681

In [12]:
# Select the first 3 real chunks (list of Documents)
sample_chunks = real_chunks[:3]

# Generate embeddings for those chunks
sample_vectors = embeddings.embed_documents([chunk.page_content for chunk in sample_chunks])

# Loop through the pairs of Document and its vector
for i, (chunk, vector) in enumerate(zip(sample_chunks, sample_vectors), start=1):
    print(f'Chunk {i} text preview: {chunk.page_content[:80]}...')
    print(f'Vector length: {len(vector)} numbers')
    print(f'First 10 numbers: {vector[:10]}')
    print('-' * 60)

Chunk 1 text preview: Agriculture in Nigeria: Crops, Livestock, and Disease Management



Agriculture ...
Vector length: 1536 numbers
First 10 numbers: [-0.007224807050079107, -0.023577062413096428, -0.010526641272008419, -0.011206623166799545, -0.007944018580019474, 0.004544109106063843, -0.023237071931362152, 0.01205660030245781, -0.04668336734175682, -0.021955566480755806]
------------------------------------------------------------
Chunk 2 text preview: Introduction

            Agriculture is one of the most important sectors of th...
Vector length: 1536 numbers
First 10 numbers: [0.018130717799067497, -0.02492818422615528, -0.01373162679374218, 0.006623490713536739, 0.008307323791086674, -0.018329547718167305, -0.018627790734171867, 0.019124863669276237, -0.03260795399546623, -0.037752654403448105]
------------------------------------------------------------
Chunk 3 text preview: Major Crops

Cassava – a major staple crop.
Yam – widely grown in the middle bel...
Vector length: 15

Step 4: Similarity Search on Real Chunks

In [13]:
query = 'What are common crop diseases and how can they be controlled?'

# Search the vector store for the top 3 most relevant chunks
results = vectorstore.similarity_search(query, k=3)

# Display the results
print(f'Top {len(results)} relevant chunks for query: "{query}"\n')

for i, doc in enumerate(results, start=1):
    print(f'Chunk: {i}:')
    print(doc.page_content)
    print(f'Source: {doc.metadata.get("source", "unknown")}')
    print(f'Doc type: {doc.metadata.get("doc_type", "unknown")}')
    print('-' * 70)

Top 3 relevant chunks for query: "What are common crop diseases and how can they be controlled?"

Chunk: 1:
Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infected plants early.
Source: 04_data_ingestion_document_processing\data\agriculture.html
Doc type: html
----------------------------------------------------------------------
Chunk: 2:
Common Crop Diseases and Control

1. Cassava Mosaic Disease
   Affected crop: Cassava
   Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
   Control: Use disease-free cuttings, plant resistant varieties, and remove infected plants.
Source: 04_data_ingestion_document_processing\data\agriculture.txt
Doc type: txt
----------------------------------------------------------------------
Chunk: 3:
• The major factors responsible for crop 


In [14]:
#query = 'What are common diseases affecting Nigerians in health sector and how can they be controlled?'

query = 'What country is the giant of Africa?. List the top causes of death in Nigeria'

# Search the vector store for the top 3 most relevant chunks
results = vectorstore.similarity_search(query, k=1)

# Display the results
print(f'Top {len(results)} relevant chunks for query: "{query}"\n')

for i, doc in enumerate(results, start=1):
    print(f'Chunk: {i}:')
    print(doc.page_content)
    print(f'Source: {doc.metadata.get("source", "unknown")}')
    print(f'Doc type: {doc.metadata.get("doc_type", "unknown")}')
    print('-' * 70)

Top 1 relevant chunks for query: "What country is the giant of Africa?. List the top causes of death in Nigeria"

Chunk: 1:
major health problem in Nigeria.  
 The top causes of death in Nigeria are; malaria, 
lower respiratory infections, HIV/AIDS,      
diarrheal diseases, road injuries, protein -energy 
malnutrition, cancer, meningitis, stroke and 
tuberculosis. 
 Malaria remains the foremost killer disease in
Source: 04_data_ingestion_document_processing\data\nigeria_health_diseases_and_prevention.pdf
Doc type: pdf
----------------------------------------------------------------------


In [15]:
query = 'List Disease Identification'

# Search the vector store for the top 3 most relevant chunks
results = vectorstore.similarity_search(query, k=3)

# Display the results
print(f'Top {len(results)} relevant chunks for query: "{query}"\n')

for i, doc in enumerate(results, start=1):
    print(f'Chunk: {i}:')
    print(doc.page_content)
    print(f'Source: {doc.metadata.get("source", "unknown")}')
    print(f'Doc type: {doc.metadata.get("doc_type", "unknown")}')
    print('-' * 70)

Top 3 relevant chunks for query: "List Disease Identification"

Chunk: 1:
Disease Identification
O Modifications brought about by the 
disease are called symptoms. 
O Structural modifications provide guide 
to the identification of the disease 
type of the causative organism.
Source: 04_data_ingestion_document_processing\data\crop_disease.pdf
Doc type: pdf
----------------------------------------------------------------------
Chunk: 2:
scientific database sources, web search engines, direct observation and relevant documents from the Nigerian Ministry 
of Health. The major public health challenges Nigeria faces are infectious diseases, control of vector some diseases,
Source: 04_data_ingestion_document_processing\data\nigeria_health_diseases_and_prevention.pdf
Doc type: pdf
----------------------------------------------------------------------
Chunk: 3:
12. Elvis EI, Akinola AF, Ikeoluwapo OA. An    
overview of disease surveillance and notification 
system in Nigeria and the roles of 

Rebuild Store, Filter Duplicates, Search

In [16]:
# 1. Rebuild the vector store using the real chunks
vectorstore = Chroma.from_documents(
    documents=real_chunks,
    embedding=embeddings,
    persist_directory=None
)

# Function to remove duplicate chunks from search results
def unique_results(results):
    seen = set()
    unique = []
    
    for doc in results:
        text = doc.page_content.strip()
        if text not in seen:
            seen.add(text)
            unique.append(doc)
            
    return unique

# Perform a query and filter duplicates
query = 'Define the following terms: Necrosis, Hypertrophy and Chlorosis'
raw_results = vectorstore.similarity_search(query, k=5)

# Remove duplicates
filtered_results = unique_results(raw_results)

# Display the unique results
print(f'Top {len(filtered_results)} unique relevant chunks for query: "{query}"\n')

for i, doc in enumerate(filtered_results, start=1):
    print(f'Chunk {i}:')
    print(doc.page_content)
    print(f'Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 90)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Top 3 unique relevant chunks for query: "Define the following terms: Necrosis, Hypertrophy and Chlorosis"

Chunk 1:
Necrosis
O This symptom appears as dead patches or 
spots all over the leaves of affected plants. It 
leads to distortion and death of organs, 
especially the leaves.
O It is caused by the attacks of fungi, bacteria 
and viruses.
Source: 04_data_ingestion_document_processing\data\crop_disease.pdf
------------------------------------------------------------------------------------------
Chunk 2:
Chlorosis
O This is a discolouration of tissues arising 
from nutrient deficiency or pathogenic 
attacks which destroys the chloroplasts and 
reduce chlorophyll quality.
Source: 04_data_ingestion_document_processing\data\crop_disease.pdf
------------------------------------------------------------------------------------------
Chunk 3:
Hypertrophy
O This refers to the enlargement of organs as 
a result of accelerated multiplication of cells 
that are under attack. 
O This causes tu

Save Vector Store to Disk

In [17]:
# Save the vector store to a folder dir
chroma_persist_dir = '../../05_embeddings_vector_stores/chroma_db'

# Rebuild the vector store with a persist dir
vectorstore_persisted = Chroma.from_documents(
    documents=real_chunks,
    embedding=embeddings,
    persist_directory=chroma_persist_dir,
)

print(f'Vector store persisted to {vectorstore_persisted}')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store persisted to <langchain_chroma.vectorstores.Chroma object at 0x0000020DD308FCD0>


Load the Persisted Vector Store

In [18]:
# Load the persisted vector store from disk
loaded_vectorstore = Chroma(
    persist_directory=chroma_persist_dir,
    embedding_function=embeddings
)
print('Loaded vector store from disk.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loaded vector store from disk.


Test Search on Loaded Store

In [19]:
# Run a query on the loaded store
query = 'What are common crop diseases?'
results = loaded_vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results, start=1):
    print(f'Chunk {i}:')
    print(doc.page_content)
    print('-' * 50)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Chunk 1:
Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infected plants early.
--------------------------------------------------
Chunk 2:
Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infected plants early.
--------------------------------------------------


In [20]:
chroma_persist_dir = 'chroma_db'

import re

def normalise_text(text: str) -> str:
    """Replace all whitespace with single spaces and strip."""
    return re.sub(r'\s+', ' ', text).strip()

# Create a set to track unique normalised texts
seen = set()
deduped_chunks = []

for chunk in real_chunks:
    norm = normalise_text(chunk.page_content)
    if norm not in seen:
        seen.add(norm)
        deduped_chunks.append(chunk)

print(f'Original chunks: {len(real_chunks)}')
print(f'Deduplicated chunks: {len(deduped_chunks)}')

Original chunks: 312
Deduplicated chunks: 312


In [21]:
# Build vector store from deduplicated chunks and persist to disk
vectorstore = Chroma.from_documents(
    documents=real_chunks,       # use deduplicated list
    embedding=embeddings,
    persist_directory=chroma_persist_dir
)

print(f'✅ Vector store persisted at {chroma_persist_dir}')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector store persisted at chroma_db


In [22]:
# import os
# print(os.listdir(chroma_persist_dir))

In [23]:
# Load the persisted store
loaded_store = Chroma(
    persist_directory=chroma_persist_dir,
    embedding_function=embeddings
)

# Search a query
query = 'What are common crop diseases?'
results = loaded_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f'Chunk {i}:')
    print(doc.page_content)
    print(f'Source: {doc.metadata.get("source", "unknown")}')
    print(f'Doc type: {doc.metadata.get("doc_type", "unknown")}')
    print('-' * 50)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Chunk 1:
Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infected plants early.
Source: 04_data_ingestion_document_processing\data\agriculture.html
Doc type: html
--------------------------------------------------
Chunk 2:
Common Crop Diseases and Control

1. Cassava Mosaic Disease
   Affected crop: Cassava
   Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
   Control: Use disease-free cuttings, plant resistant varieties, and remove infected plants.
Source: 04_data_ingestion_document_processing\data\agriculture.txt
Doc type: txt
--------------------------------------------------
Chunk 3:
• The major factors responsible for crop 
diseases are fungi, bacteria, viruses and 
nematodes. 
• Others are nutrient deficiencies and air 
pollutants. 
• The severity of crop diseas